In [0]:

gold = spark.table("workspace.gold.amostra_wide")

print("Total de registros:", gold.count())
print("Total de colunas:", len(gold.columns))

display(
    gold.select(
        "usuario_id",
        "destino",
        "avaliacao",
        "form_b_pt",
        "form_j_pt",
        "orcamento_todas_faixas",
        "popularidade_todas_opcoes",
        "regiao_inconsistente",
        "status_regiao"
    ).limit(20)
)


In [0]:

from pyspark.sql import functions as F

# Carregar Silver auditada
perfis = spark.table(
    "workspace.silver.perfis_auditados"
)

# Carregar avaliações consolidadas
avaliacoes = spark.table(
    "workspace.silver.avaliacoes_consolidadas"
)

# Selecionar informações dos destinos
avaliacoes_base = avaliacoes.select(
    "usuario_id",
    "destino_id",
    "destino",
    "cidade",
    "pais",
    "avaliacao"
)

# Integrar avaliações e perfis auditados
gold_nova = (
    avaliacoes_base
    .join(
        perfis,
        avaliacoes_base.usuario_id == perfis.id,
        "left"
    )
    .withColumn(
        "target",
        F.when(F.col("avaliacao") == "Sim", 1)
         .when(F.col("avaliacao") == "Não", 0)
         .otherwise(F.lit(None).cast("int"))
    )
)

# Conferir antes de salvar
print("Registros:", gold_nova.count())
print("Colunas:", len(gold_nova.columns))

# Salvar somente se a estrutura estiver correta
assert gold_nova.count() == 813802
assert len(gold_nova.columns) == 72

(
    gold_nova.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.amostra_wide")
)

print("Gold atualizada com sucesso!")


In [0]:

from pyspark.sql import functions as F

gold = spark.table("workspace.gold.amostra_wide")

print("Total de avaliações:", gold.count())
print("Total de colunas:", len(gold.columns))

print(
    "Viajantes distintos:",
    gold.select("usuario_id").distinct().count()
)

print(
    "Destinos distintos:",
    gold.select("destino_id").distinct().count()
)

print(
    "Registros sem perfil associado:",
    gold.filter(F.col("id").isNull()).count()
)

display(
    gold.groupBy("avaliacao", "target")
        .count()
        .orderBy("avaliacao")
)


In [0]:

from pyspark.sql import functions as F

gold = spark.table("workspace.gold.amostra_wide")

print("Total de registros:", gold.count())
print("Total de colunas:", len(gold.columns))
print("Viajantes distintos:", gold.select("usuario_id").distinct().count())
print("Destinos distintos:", gold.select("destino_id").distinct().count())

print("Distribuição das avaliações:")
display(
    gold.groupBy("avaliacao", "target")
        .count()
        .orderBy("avaliacao")
)

print("Registros sem perfil associado:")
print(
    gold.filter(F.col("id").isNull()).count()
)


In [0]:
from pyspark.sql import functions as F

perfis = spark.table("workspace.silver.perfis_auditados")

avaliacoes = spark.table("workspace.silver.avaliacoes_consolidadas")

# Selecionar apenas as informações de avaliação
# para evitar colunas duplicadas
avaliacoes_base = avaliacoes.select(
    "usuario_id", "destino_id", "destino", "cidade", "pais", "avaliacao"
)

# Juntar os perfis completos às avaliações
gold = avaliacoes_base.join(
    perfis, avaliacoes_base.usuario_id == perfis.id, "left"
).withColumn(
    "target",
    F.when(F.col("avaliacao") == "Sim", 1)
    .when(F.col("avaliacao") == "Não", 0)
    .otherwise(F.lit(None).cast("int")),
)

# Salvar Gold completa
(
    gold.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.amostra_wide")
)

print("Total de avaliações:", gold.count())
print("Total de colunas:", len(gold.columns))

display(
    gold.select(
        "usuario_id",
        "destino",
        "pais",
        "form_a_pt",
        "form_b_pt",
        "form_c_pt",
        "form_f_pt",
        "form_g_pt",
        "avaliacao",
        "target",
    )
)